In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_kddcup99
# Using 10% of the dataset to keep the experiments manageable.
data = fetch_kddcup99( percent10=True, shuffle=True, random_state=42, download_if_missing=True )


print("Features shape:", data.data.shape)
print("Labels shape:", data.target.shape)




Features shape: (494021, 41)
Labels shape: (494021,)


In [ ]:
# features
X = pd.DataFrame(data.data, columns=data.feature_names)
# labels
y = pd.Series(data.target, name="label")
display(X.head())

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
0,0,b'icmp',b'ecr_i',b'SF',1032,0,0,0,0,0,...,255,255,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0,b'icmp',b'ecr_i',b'SF',1032,0,0,0,0,0,...,255,255,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0,b'icmp',b'ecr_i',b'SF',1032,0,0,0,0,0,...,255,255,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0,b'tcp',b'http',b'SF',345,5419,0,0,0,0,...,101,255,1.0,0.0,0.01,0.02,0.0,0.0,0.0,0.0
4,0,b'tcp',b'private',b'S0',0,0,0,0,0,0,...,255,2,0.01,0.07,0.0,0.0,1.0,1.0,0.0,0.0


In [ ]:
# Grouping labels into normal traffic and attacks
normal_count = (y == b'normal.').sum()
attack_count = (y != b'normal.').sum()
print("Normal:", normal_count)
print("Attacks:", attack_count)

Normal: 97278
Attacks: 396743


In [ ]:
cat_cols = ["protocol_type", "service", "flag"]
num_cols = [col for col in X.columns if col not in cat_cols]

# These columns were loaded as objects,so converted them before modeling
X[num_cols] = X[num_cols].apply(pd.to_numeric)

In [ ]:
# 0 = normal
# 1 = attack
y_binary = (y != b'normal.').astype(int)
y_binary.value_counts().sort_index()

,count
label,
0,97278
1,396743


In [ ]:
from sklearn.model_selection import train_test_split

# Keeping class proportions similar across all three splits
X_train_raw, X_remaining, y_train, y_remaining = train_test_split(
    X, y_binary,
    test_size=0.30,
    random_state=42,
    stratify=y_binary
)

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_remaining, y_remaining,
    test_size=0.50,
    random_state=42,
    stratify=y_remaining
)

# Verifying splits work as we expect them to
print("Train:", X_train_raw.shape)
print("Validation:", X_val_raw.shape)
print("Test:", X_test_raw.shape)

Train: (345814, 41)
Validation: (74103, 41)
Test: (74104, 41)


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

# Fitting only on training data to avoid data leakage
X_train_cat = encoder.fit_transform(X_train_raw[cat_cols])
X_val_cat = encoder.transform(X_val_raw[cat_cols])
X_test_cat = encoder.transform(X_test_raw[cat_cols])

# Checking that all splits have same encoded features
print("Train:", X_train_cat.shape)
print("Validation:", X_val_cat.shape)
print("Test:", X_test_cat.shape)

Train: (345814, 80)
Validation: (74103, 80)
Test: (74104, 80)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Using only normal traffic to fit scaler
normal_mask = y_train == 0
X_train_num_raw = X_train_raw[num_cols]

scaler = StandardScaler()
scaler.fit(X_train_num_raw.loc[normal_mask])

# Applying same scaling to all three splits
X_train_num = scaler.transform(X_train_num_raw)
X_val_num = scaler.transform(X_val_raw[num_cols])
X_test_num = scaler.transform(X_test_raw[num_cols])

print("Train:", X_train_num.shape)
print("Validation:", X_val_num.shape)
print("Test:", X_test_num.shape)

Train: (345814, 38)
Validation: (74103, 38)
Test: (74104, 38)


In [ ]:
from scipy.sparse import csr_matrix, hstack

# Keeping everything sparse to avoid using too much memory
X_train_num_sparse = csr_matrix(X_train_num)
X_val_num_sparse = csr_matrix(X_val_num)
X_test_num_sparse = csr_matrix(X_test_num)

# Combining numerical and categorical features for the model
X_train_processed = hstack([X_train_num_sparse, X_train_cat], format="csr")
X_val_processed = hstack([X_val_num_sparse, X_val_cat], format="csr")
X_test_processed = hstack([X_test_num_sparse, X_test_cat], format="csr")

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Train: (345814, 118)
Validation: (74103, 118)
Test: (74104, 118)


In [ ]:
# Keeping only normal traffic to learn what usual behavior looks like
normal_mask = (y_train.to_numpy() == 0)
X_train_normal = X_train_processed[normal_mask]

print("Normal training data:", X_train_normal.shape)

Normal training data: (68094, 118)


In [ ]:
# Convert to float32 to keep memory usage lower.
X_train_normal_dense = X_train_normal.toarray().astype(np.float32)

print("Shape:", X_train_normal_dense.shape)
print("Memory usage:", round(X_train_normal_dense.nbytes / (1024 ** 2), 2), "MB")

Shape: (68094, 118)
Memory usage: 30.65 MB


In [ ]:
from sklearn.decomposition import IncrementalPCA

pca = IncrementalPCA(n_components=20, batch_size=4096)

# Processing the data in batches to keep memory usage manageable
pca.fit(X_train_normal_dense)

print("Explained variance:", pca.explained_variance_ratio_.sum())

Explained variance: 0.8930969256331315


In [ ]:
explained_variance = pca.explained_variance_ratio_

print("Explained variance (%):", round(explained_variance.sum() * 100, 2))

Explained variance (%): 89.31


In [ ]:
import numpy as np

def reconstruction_errors(X, pca, batch_size=4096):
    errors = []

    # Processing batches to avoid converting the entire dataset to dense format
    for start in range(0, X.shape[0], batch_size):
        X_batch = X[start:start + batch_size].toarray().astype(np.float32)

        X_reduced = pca.transform(X_batch)
        X_reconstructed = pca.inverse_transform(X_reduced)

        batch_errors = np.mean((X_batch - X_reconstructed) ** 2, axis=1)
        errors.extend(batch_errors)

    return np.array(errors)

In [ ]:
val_errors = reconstruction_errors(X_val_processed, pca)

print("Validation records:", len(val_errors))
print("Median reconstruction error:", np.median(val_errors))
print("Maximum reconstruction error:", val_errors.max())

Validation records: 74103
Median reconstruction error: 0.26819953604356206
Maximum reconstruction error: 29.031030520227233


In [ ]:
# Setting the threshold using normal validation traffic only
normal_mask = y_val.to_numpy() == 0
normal_val_errors = val_errors[normal_mask]

threshold = np.percentile(normal_val_errors, 95)

print("Normal validation records:", len(normal_val_errors))
print("Anomaly threshold:", threshold)

Normal validation records: 14592
Anomaly threshold: 0.06850316060984525


In [ ]:
test_errors = reconstruction_errors(X_test_processed, pca)

print("Test records:", len(test_errors))
print("Median reconstruction error:", np.median(test_errors))
print("Maximum reconstruction error:", test_errors.max())

Test records: 74104
Median reconstruction error: 0.26819953604356206
Maximum reconstruction error: 6681.9402254519855


In [ ]:
# Flag records with errors above threshold as attacks
y_pred = (test_errors > threshold).astype(int)

print("Predicted normal:", (y_pred == 0).sum())
print("Predicted attacks:", (y_pred == 1).sum())

Predicted normal: 14266
Predicted attacks: 59838


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

y_test_array = y_test.to_numpy()

print(confusion_matrix(y_test_array, y_pred))
print(classification_report(y_test_array, y_pred, target_names=["Normal", "Attack"], digits=4))
print("Average Precision:", round(average_precision_score(y_test_array, test_errors), 4))

[[13891   701]
 [  375 59137]]
              precision    recall  f1-score   support

      Normal     0.9737    0.9520    0.9627     14592
      Attack     0.9883    0.9937    0.9910     59512

    accuracy                         0.9855     74104
   macro avg     0.9810    0.9728    0.9768     74104
weighted avg     0.9854    0.9855    0.9854     74104

Average Precision: 0.9965


In [ ]:
# Ensuring that the splits don't share records
val_idx = set(X_val_raw.index)
test_idx = set(X_test_raw.index)
train_idx = set(X_train_raw.index)

print("Val-test overlap:", len(val_idx & test_idx))
print("Train-val overlap:", len(train_idx & val_idx))
print("Train-test overlap:", len(train_idx & test_idx))

print("Validation and test errors identical:", np.array_equal(val_errors, test_errors))

Val-test overlap: 0
Train-val overlap: 0
Train-test overlap: 0
Validation and test errors identical: False


In [ ]:
import numpy as np

# Comparing error distributions across validation and test sets
for name, errors in [("Validation", val_errors), ("Test", test_errors)]:
    print(f"\n{name}")
    print("Mean:", np.mean(errors))
    print("Median:", np.median(errors))
    print("90th percentile:", np.percentile(errors, 90))
    print("95th percentile:", np.percentile(errors, 95))
    print("99th percentile:", np.percentile(errors, 99))

print("\nExact median equality:", np.median(val_errors) == np.median(test_errors))


Validation
Mean: 1.354123107762063
Median: 0.26819953604356206
90th percentile: 6.324487883525293
95th percentile: 6.704622046892383
99th percentile: 6.963307869861098

Test
Mean: 1.459112340805425
Median: 0.26819953604356206
90th percentile: 6.35469577367049
95th percentile: 6.7074009361463345
99th percentile: 6.957888168974379

Exact median equality: True


In [ ]:
# Checking whether reconstruction errors repeat across records
val_values, val_counts = np.unique(val_errors, return_counts=True)
test_values, test_counts = np.unique(test_errors, return_counts=True)

print("Unique validation errors:", len(val_values))
print("Unique test errors:", len(test_values))

# Check the most frequent values
for name, values, counts in [
    ("Validation", val_values, val_counts),
    ("Test", test_values, test_counts)
]:
    order = np.argsort(counts)[::-1]
    print(f"\n{name} - five most common errors:")
    for i in order[:5]:
        print(f"{values[i]:.10f}: {counts[i]}")

Unique validation errors: 27466
Unique test errors: 27498

Validation - five most common errors:
0.2681995360: 29021
0.2682035721: 4961
0.2674655848: 3907
0.2667330810: 776
0.2660020245: 182

Test - five most common errors:
0.2681995360: 28904
0.2682035721: 5006
0.2674655848: 3890
0.2667330810: 796
0.2660020245: 201


In [ ]:
y_val_array = y_val.to_numpy()

normal_errors = val_errors[y_val_array == 0]
attack_errors = val_errors[y_val_array == 1]

print("Normal median:", np.median(normal_errors))
print("Attack median:", np.median(attack_errors))
print("Normal 95th percentile:", np.percentile(normal_errors, 95))
print("Attacks above threshold (%):", round(np.mean(attack_errors > threshold) * 100, 2))

Normal median: 0.003923505402243225
Attack median: 0.26819953604356206
Normal 95th percentile: 0.06850316060984525
Attacks above threshold (%): 99.38


In [ ]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test.to_numpy(), y_pred).ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print(f"False positive rate: {fpr:.4%}")
print(f"False negative rate: {fnr:.4%}")

False positive rate: 4.8040%
False negative rate: 0.6301%


In [ ]:
# Check whether identical feature rows appear in different splits
train_rows = set(pd.util.hash_pandas_object(X_train_raw, index=False))
val_rows = set(pd.util.hash_pandas_object(X_val_raw, index=False))
test_rows = set(pd.util.hash_pandas_object(X_test_raw, index=False))

print("Train-val duplicates:", len(train_rows & val_rows))
print("Train-test duplicates:", len(train_rows & test_rows))
print("Val-test duplicates:", len(val_rows & test_rows))

Train-val duplicates: 7896
Train-test duplicates: 7865
Val-test duplicates: 4029


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

# Check which test rows don't appear in training
train_hashes = pd.util.hash_pandas_object(X_train_raw, index=False)
test_hashes = pd.util.hash_pandas_object(X_test_raw, index=False)
unseen_mask = ~test_hashes.isin(set(train_hashes))

print("Original test records:", len(X_test_raw))
print("Unseen test records:", unseen_mask.sum())
print("Retained (%):", round(unseen_mask.mean() * 100, 2))

# Recheck performance on unseen rows only
y_unseen = y_test.to_numpy()[unseen_mask]
errors_unseen = test_errors[unseen_mask]
pred_unseen = (errors_unseen > threshold).astype(int)

print("\nConfusion matrix:")
print(confusion_matrix(y_unseen, pred_unseen, labels=[0, 1]))

print("\nClassification report:")
print(classification_report(
    y_unseen, pred_unseen,
    labels=[0, 1],
    target_names=["Normal", "Attack"],
    digits=4,
    zero_division=0
))

print("Average precision:", round(average_precision_score(y_unseen, errors_unseen), 4))

Original test records: 74104
Unseen test records: 19910
Retained (%): 26.87

Confusion matrix:
[[12364   616]
 [  177  6753]]

Classification report:
              precision    recall  f1-score   support

      Normal     0.9859    0.9525    0.9689     12980
      Attack     0.9164    0.9745    0.9445      6930

    accuracy                         0.9602     19910
   macro avg     0.9511    0.9635    0.9567     19910
weighted avg     0.9617    0.9602    0.9604     19910

Average precision: 0.986
